# Phylogenetics: Reconstructing Evolutionary History

Phylogenetic trees are hypotheses about the evolutionary relationships among organisms, genes, or other biological entities. This notebook covers the theory, algorithms, and practical tools for building and interpreting phylogenetic trees.

---

## Learning Objectives

- Master phylogenetic tree terminology (nodes, branches, clades, rooted vs unrooted)
- Implement distance-based methods from scratch: UPGMA and Neighbor-Joining
- Understand distance matrices and models of sequence evolution (Jukes-Cantor, Kimura)
- Grasp character-based and probabilistic methods (Parsimony, Maximum Likelihood, Bayesian)
- Perform and interpret bootstrap analysis
- Build and visualize trees with BioPython (Bio.Phylo)
- Interpret phylogenetic trees in biological context

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
The methods here are applied in sequence analysis, annotation, motif/protein interpretation, and evidence building from biological databases.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Similarity is not identity: alignments, scores, and biological function are related but not equivalent.
- Database provenance matters: always track which release/version generated your result.
- Threshold choices (e-value, identity, score) strongly change sensitivity vs specificity.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

[← Previous: Multiple Sequence Alignment: From Theory to Practice](../../Tier_2_Core_Bioinformatics/05_Multiple_Sequence_Alignment/01_multiple_sequence_alignment.ipynb) | [Next: Protein Structure Analysis →](../../Tier_2_Core_Bioinformatics/07_Protein_Structure/01_protein_structure.ipynb)

---

---

## 1. What is a Phylogenetic Tree?

A phylogenetic tree (or phylogeny) represents the inferred evolutionary relationships among a set of organisms or sequences. Charles Darwin included only a single illustration in *On the Origin of Species* -- a phylogenetic tree.

### Anatomy of a Phylogenetic Tree

```
                         ROOT (Last Common Ancestor)
                           |
              +------------+------------+
              |                         |
         Internal Node             Internal Node
              |                         |
        +-----+-----+             +-----+-----+
        |           |             |           |
      Human      Chimp         Mouse        Rat    <-- Tips (Leaves/OTUs)

   Terminology:
   - Node:        Point where branches meet or terminate
   - Branch:      Line connecting nodes (length = evolutionary change)
   - Tip/Leaf:    Terminal node (extant organism or sequence)
   - OTU:         Operational Taxonomic Unit (generic for tip)
   - Internal node: Hypothetical ancestor (not directly observed)
   - Root:        Last common ancestor (LCA) of all tips
   - Clade:       A node and ALL its descendants
   - Sister taxa: Taxa sharing an immediate common ancestor
```

### Tree Types

| Type | Description | Branch Lengths |
|------|-------------|----------------|
| **Rooted tree** | Has a direction of time; root = LCA | Meaningful |
| **Unrooted tree** | Shows relationships but not ancestry direction | Meaningful |
| **Cladogram** | Shows topology only | Not meaningful |
| **Phylogram** | Branch lengths proportional to evolutionary change | Proportional |
| **Ultrametric tree** | All tips equidistant from root (molecular clock) | Time-proportional |

In [ ]:
class PhyloNode:
    """
    A node in a phylogenetic tree.
    """
    def __init__(self, name=None, branch_length=0.0):
        self.name = name
        self.branch_length = branch_length
        self.children = []
        self.parent = None
    
    def add_child(self, child):
        child.parent = self
        self.children.append(child)
        return child
    
    def is_leaf(self):
        return len(self.children) == 0
    
    def get_leaves(self):
        if self.is_leaf():
            return [self]
        leaves = []
        for child in self.children:
            leaves.extend(child.get_leaves())
        return leaves
    
    def to_newick(self):
        if self.is_leaf():
            return f"{self.name}:{self.branch_length:.4f}"
        children_str = ','.join(c.to_newick() for c in self.children)
        if self.parent is None:  # root
            return f"({children_str});"
        return f"({children_str}):{self.branch_length:.4f}"


# Build a simple tree: ((Human:0.1, Chimp:0.12):0.2, (Mouse:0.25, Rat:0.23):0.15)
root = PhyloNode()

primate = root.add_child(PhyloNode(branch_length=0.2))
primate.add_child(PhyloNode('Human', 0.1))
primate.add_child(PhyloNode('Chimp', 0.12))

rodent = root.add_child(PhyloNode(branch_length=0.15))
rodent.add_child(PhyloNode('Mouse', 0.25))
rodent.add_child(PhyloNode('Rat', 0.23))

print("Tree leaves:", [n.name for n in root.get_leaves()])
print("Newick:", root.to_newick())

### Why Build Phylogenies?

Phylogenetic trees are not just academic exercises. They are essential tools for:

1. **Understanding evolutionary relationships**: Which species are most closely related?
2. **Gene family analysis**: Understanding gene duplication, loss, and functional divergence (e.g., the globin gene family: hemoglobin, myoglobin)
3. **Disease tracking**: Tracing viral evolution in real-time (e.g., SARS-CoV-2 phylogenomics)
4. **Ancestral sequence reconstruction**: Inferring what extinct proteins looked like
5. **Horizontal gene transfer detection**: Identifying genes that jump between species
6. **Community ecology**: Comparing microbiome compositions across samples

---

## 2. The Newick Format

The standard text representation of phylogenetic trees. Understanding Newick is essential for working with any phylogenetic software.

```
Format examples:

  (A,B,C);                          Star tree (all connected to root)
  ((A,B),C);                        A and B are sisters, C is outgroup
  ((A:0.1,B:0.2):0.3,C:0.4);       With branch lengths
  ((A:0.1,B:0.2)95:0.3,C:0.4);     With bootstrap support value (95%)

  Visual:           Newick:
       +--A         ((A:0.1,B:0.2):0.3,(C:0.15,D:0.15):0.25);
    +--+
    |  +--B
  --+
    |  +--C
    +--+
       +--D
```

In [ ]:
import re

def parse_newick(newick):
    """
    Parse a Newick string into a PhyloNode tree.
    Handles names, branch lengths, and nested clades.
    """
    newick = newick.strip().rstrip(';')
    
    def _parse(s, pos=0):
        node = PhyloNode()
        children = []
        
        if s[pos] == '(':
            pos += 1  # skip '('
            while True:
                child, pos = _parse(s, pos)
                children.append(child)
                if pos < len(s) and s[pos] == ',':
                    pos += 1
                elif pos < len(s) and s[pos] == ')':
                    pos += 1
                    break
                else:
                    break
            for child in children:
                node.add_child(child)
        
        # Parse name and branch length
        name = ''
        while pos < len(s) and s[pos] not in ',):;':
            name += s[pos]
            pos += 1
        
        if ':' in name:
            parts = name.split(':')
            node.name = parts[0] if parts[0] else None
            node.branch_length = float(parts[1]) if parts[1] else 0.0
        elif name:
            node.name = name
        
        return node, pos
    
    tree, _ = _parse(newick, 0)
    return tree


def print_tree_ascii(node, prefix="", is_last=True, is_root=True):
    """
    Print an ASCII representation of the tree.
    """
    connector = "" if is_root else ("\u2514\u2500 " if is_last else "\u251c\u2500 ")
    name = node.name or ""
    bl = f":{node.branch_length:.4f}" if node.branch_length > 0 else ""
    print(f"{prefix}{connector}{name}{bl}")
    
    child_prefix = prefix + ("   " if is_last or is_root else "\u2502  ")
    for i, child in enumerate(node.children):
        print_tree_ascii(child, child_prefix, i == len(node.children) - 1, False)


# Parse and display
newick_str = "((Human:0.1,Chimp:0.12):0.2,(Mouse:0.25,Rat:0.23):0.15);"
tree = parse_newick(newick_str)

print(f"Newick: {newick_str}")
print(f"\nTree structure:")
print_tree_ascii(tree)

---

## 3. Distance Matrices and Models of Evolution

### 3.1 From Sequences to Distances

The simplest distance between two aligned sequences is the **p-distance** -- the proportion of sites that differ:

$$p = \frac{\text{number of differences}}{\text{number of compared sites}}$$

However, p-distance **underestimates** the true evolutionary distance because it ignores **multiple substitutions** at the same site (back-mutations, parallel mutations).

### 3.2 The Jukes-Cantor Model (JC69)

The simplest substitution model assumes:
- All substitutions are equally likely
- All nucleotides have equal frequency (25% each)

The JC69-corrected distance:

$$d = -\frac{3}{4} \ln\left(1 - \frac{4p}{3}\right)$$

This correction accounts for unobserved substitutions. Note that $d$ is undefined when $p \geq 0.75$ (sequences are so divergent that the model breaks down).

### 3.3 The Kimura Two-Parameter Model (K2P)

A more realistic model that distinguishes **transitions** (purine-to-purine: A<->G, or pyrimidine-to-pyrimidine: C<->T) from **transversions** (purine-to-pyrimidine and vice versa):

$$d = -\frac{1}{2} \ln(1 - 2S - V) - \frac{1}{4} \ln(1 - 2V)$$

Where $S$ = proportion of transitions and $V$ = proportion of transversions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def p_distance(seq1, seq2):
    """Proportion of differing sites (ignoring gaps)."""
    diffs = 0
    compared = 0
    for a, b in zip(seq1, seq2):
        if a != '-' and b != '-':
            compared += 1
            if a != b:
                diffs += 1
    return diffs / compared if compared > 0 else 0


def jukes_cantor_distance(seq1, seq2):
    """JC69-corrected distance."""
    p = p_distance(seq1, seq2)
    if p >= 0.75:
        return float('inf')  # Sequences too divergent
    return -0.75 * np.log(1 - 4 * p / 3)


def kimura_distance(seq1, seq2):
    """Kimura 2-parameter corrected distance."""
    transitions = 0
    transversions = 0
    compared = 0
    
    purines = set('AG')
    pyrimidines = set('CT')
    
    for a, b in zip(seq1, seq2):
        if a != '-' and b != '-':
            compared += 1
            if a != b:
                if (a in purines and b in purines) or (a in pyrimidines and b in pyrimidines):
                    transitions += 1
                else:
                    transversions += 1
    
    if compared == 0:
        return 0
    
    S = transitions / compared
    V = transversions / compared
    
    term1 = 1 - 2 * S - V
    term2 = 1 - 2 * V
    
    if term1 <= 0 or term2 <= 0:
        return float('inf')
    
    return -0.5 * np.log(term1) - 0.25 * np.log(term2)


# Compare distance metrics
seq_a = "ATCGATCGATCGATCGATCG"
seq_b = "ATCGTTCGATCGATCGATCG"  # 1 transition
seq_c = "TTCGATCGATCGATCAATCG"  # 1 transversion + 1 transversion

print("Distance comparison:")
print(f"{'Pair':<15} {'p-distance':>12} {'JC69':>12} {'Kimura':>12}")
print("-" * 55)

for name, s1, s2 in [("A vs B", seq_a, seq_b), ("A vs C", seq_a, seq_c), ("B vs C", seq_b, seq_c)]:
    pd = p_distance(s1, s2)
    jc = jukes_cantor_distance(s1, s2)
    k2 = kimura_distance(s1, s2)
    print(f"{name:<15} {pd:>12.4f} {jc:>12.4f} {k2:>12.4f}")

In [ ]:
# Visualize how corrections diverge from p-distance
p_values = np.arange(0.01, 0.74, 0.01)
jc_values = [-0.75 * np.log(1 - 4 * p / 3) for p in p_values]

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(p_values, p_values, 'b-', linewidth=2, label='p-distance (observed)')
ax.plot(p_values, jc_values, 'r-', linewidth=2, label='JC69-corrected (estimated actual)')
ax.plot([0, 0.75], [0, 0.75], 'k:', alpha=0.3)

ax.set_xlabel('Observed p-distance', fontsize=12)
ax.set_ylabel('Estimated evolutionary distance', fontsize=12)
ax.set_title('Jukes-Cantor Correction: Why p-distance Underestimates', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 0.75)
ax.set_ylim(0, 2.0)
ax.grid(True, alpha=0.3)

# Annotate the divergence
ax.annotate('Multiple substitutions\nbecome significant',
            xy=(0.5, 0.5), xytext=(0.3, 1.2),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10, color='gray')

plt.tight_layout()
plt.show()

print("Key insight: at p=0.30, JC69 estimates d=0.38 -- 27% more than observed.")
print("At p=0.50, JC69 estimates d=0.82 -- 64% more than observed.")
print("At p>=0.75, saturation -- random expectation for 4 equally likely nucleotides.")

In [ ]:
def build_distance_matrix(sequences, names, method='jukes-cantor'):
    """
    Calculate pairwise distance matrix from aligned sequences.
    """
    n = len(sequences)
    matrix = np.zeros((n, n))
    
    distance_fn = {
        'p-distance': p_distance,
        'jukes-cantor': jukes_cantor_distance,
        'kimura': kimura_distance,
    }[method]
    
    for i in range(n):
        for j in range(i + 1, n):
            d = distance_fn(sequences[i], sequences[j])
            matrix[i, j] = matrix[j, i] = d
    
    return matrix


def print_distance_matrix(matrix, names):
    """Pretty-print a distance matrix."""
    n = len(names)
    max_name = max(len(name) for name in names)
    
    header = ' ' * (max_name + 2)
    for name in names:
        header += f"{name:>8}"
    print(header)
    
    for i in range(n):
        row = f"{names[i]:>{max_name}}  "
        for j in range(n):
            row += f"{matrix[i, j]:>8.4f}"
        print(row)


# Example with primate and rodent sequences
aligned_seqs = [
    "ATCGATCGATCGATCGATCG",  # Species A
    "ATCGTTCGATCGATCGATCG",  # Species B (1 change from A)
    "TTCGATCGATCGATCGATCG",  # Species C (1 change from A)
    "TTCGTTCGATCGATCGATCG",  # Species D (2 changes from A)
    "AACCGGTTAACCGGTTAACC",  # Species E (very different)
]
seq_names = ["Sp_A", "Sp_B", "Sp_C", "Sp_D", "Sp_E"]

dm = build_distance_matrix(aligned_seqs, seq_names, method='jukes-cantor')
print("Distance matrix (Jukes-Cantor):")
print_distance_matrix(dm, seq_names)

In [ ]:
def plot_distance_matrix(matrix, names, title="Distance Matrix"):
    """Visualize distance matrix as a heatmap."""
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(matrix, cmap='YlOrRd', vmin=0)
    
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha='right', fontsize=10)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=10)
    
    for i in range(len(names)):
        for j in range(len(names)):
            color = 'white' if matrix[i, j] > matrix.max() * 0.6 else 'black'
            ax.text(j, i, f"{matrix[i, j]:.3f}", ha='center', va='center',
                    fontsize=9, color=color)
    
    plt.colorbar(im, label='Distance')
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    plt.show()


plot_distance_matrix(dm, seq_names, title="JC69 Distance Matrix")

---

## 4. Distance-Based Methods

Distance-based methods convert a distance matrix into a tree. They are fast but discard the original character data.

### 4.1 UPGMA (Unweighted Pair Group Method with Arithmetic Mean)

UPGMA is the simplest hierarchical clustering method for tree building.

**Algorithm:**
1. Find the smallest distance $d_{ij}$ in the matrix
2. Create a new node joining $i$ and $j$, with branch lengths $d_{ij}/2$ each
3. Compute distances to the new cluster as arithmetic mean of distances to its members
4. Replace $i$ and $j$ with the new cluster in the matrix
5. Repeat until one cluster remains

**Key assumption**: UPGMA assumes a **molecular clock** -- all lineages evolve at the same rate. This produces an ultrametric tree where all tips are equidistant from the root.

**Limitation**: When the molecular clock assumption is violated (which is common), UPGMA produces incorrect topologies.

In [ ]:
def upgma(dist_matrix, names):
    """
    UPGMA algorithm implemented from scratch.
    
    Returns a Newick-format tree string.
    """
    n = len(names)
    D = dist_matrix.copy()
    node_names = list(names)
    cluster_sizes = [1] * n
    node_heights = [0.0] * n  # Track heights for branch length calculation
    
    step = 0
    while len(node_names) > 1:
        m = len(node_names)
        step += 1
        
        # Step 1: Find minimum distance
        min_dist = np.inf
        min_i, min_j = 0, 1
        for i in range(m):
            for j in range(i + 1, m):
                if D[i, j] < min_dist:
                    min_dist = D[i, j]
                    min_i, min_j = i, j
        
        i, j = min_i, min_j
        new_height = min_dist / 2
        
        # Step 2: Calculate branch lengths from current nodes to the new internal node
        bl_i = new_height - node_heights[i]
        bl_j = new_height - node_heights[j]
        
        new_name = f"({node_names[i]}:{bl_i:.4f},{node_names[j]}:{bl_j:.4f})"
        new_size = cluster_sizes[i] + cluster_sizes[j]
        
        print(f"Step {step}: Merge {node_names[i]} + {node_names[j]} at distance {min_dist:.4f}")
        
        # Step 3: Update distance matrix
        new_D = np.zeros((m - 1, m - 1))
        new_names = []
        new_sizes = []
        new_heights = []
        idx_map = []
        
        for k in range(m):
            if k != i and k != j:
                new_names.append(node_names[k])
                new_sizes.append(cluster_sizes[k])
                new_heights.append(node_heights[k])
                idx_map.append(k)
        
        new_names.append(new_name)
        new_sizes.append(new_size)
        new_heights.append(new_height)
        
        # Copy existing distances
        for a, ka in enumerate(idx_map):
            for b, kb in enumerate(idx_map):
                new_D[a, b] = D[ka, kb]
        
        # Step 3: Distances to new cluster = weighted arithmetic mean
        for a, ka in enumerate(idx_map):
            d_new = (cluster_sizes[i] * D[ka, i] + cluster_sizes[j] * D[ka, j]) / new_size
            new_D[a, -1] = new_D[-1, a] = d_new
        
        D = new_D
        node_names = new_names
        cluster_sizes = new_sizes
        node_heights = new_heights
    
    return node_names[0] + ";"


# Apply UPGMA to our distance matrix
print("UPGMA Tree Construction:")
print("=" * 60)
upgma_newick = upgma(dm, seq_names)
print(f"\nResult: {upgma_newick}")

### Step-by-Step UPGMA Walkthrough

Let us trace through UPGMA with a concrete example to solidify understanding.

In [ ]:
# Textbook UPGMA example
example_dm = np.array([
    [0.,  4.,  2.,  5.,  6.],
    [4.,  0.,  3.,  6.,  5.],
    [2.,  3.,  0.,  3.,  4.],
    [5.,  6.,  3.,  0.,  1.],
    [6.,  5.,  4.,  1.,  0.]
])
example_names = ['s1', 's2', 's3', 's4', 's5']

print("Input distance matrix:")
print_distance_matrix(example_dm, example_names)
print()

print("Step-by-step UPGMA:")
print("=" * 60)
example_tree = upgma(example_dm, example_names)
print(f"\nFinal Newick: {example_tree}")

print("\nExpected topology: ((s1,s3),s2) groups with (s4,s5)")
print("s4 and s5 are closest (distance=1), then s1 and s3 (distance=2)")

### 4.2 Neighbor-Joining (NJ)

Neighbor-Joining is the most widely used distance-based method. Unlike UPGMA, it does **not** assume a molecular clock.

**Algorithm:**
1. Compute the **Q-matrix** from the distance matrix:
   $$Q(i,j) = (n-2) \times d(i,j) - \sum_k d(i,k) - \sum_k d(j,k)$$
2. Find the pair $(i,j)$ with the **minimum** $Q(i,j)$ and join them
3. Calculate branch lengths to the new node:
   $$d(i,u) = \frac{1}{2}d(i,j) + \frac{1}{2(n-2)}\left[\sum_k d(i,k) - \sum_k d(j,k)\right]$$
   $$d(j,u) = d(i,j) - d(i,u)$$
4. Update distances to the new node:
   $$d(u,k) = \frac{1}{2}[d(i,k) + d(j,k) - d(i,j)]$$
5. Repeat until two nodes remain

**Key advantage**: NJ allows unequal branch lengths, producing more realistic trees when evolutionary rates vary across lineages.

**Complexity**: $O(n^3)$

In [ ]:
def neighbor_joining(dist_matrix, names):
    """
    Neighbor-Joining algorithm implemented from scratch.
    
    Returns a Newick-format tree string.
    """
    n = len(names)
    D = dist_matrix.copy()
    node_names = list(names)
    
    step = 0
    while len(node_names) > 2:
        m = len(node_names)
        step += 1
        
        # Step 1: Compute Q-matrix
        r = D.sum(axis=1)  # Row sums
        Q = np.full((m, m), np.inf)
        for i in range(m):
            for j in range(i + 1, m):
                Q[i, j] = Q[j, i] = (m - 2) * D[i, j] - r[i] - r[j]
        
        # Step 2: Find minimum Q
        i, j = np.unravel_index(Q.argmin(), Q.shape)
        if i > j:
            i, j = j, i
        
        # Step 3: Calculate branch lengths
        d_iu = 0.5 * D[i, j] + (r[i] - r[j]) / (2 * (m - 2))
        d_ju = D[i, j] - d_iu
        
        # Ensure non-negative branch lengths
        d_iu = max(d_iu, 0.0)
        d_ju = max(d_ju, 0.0)
        
        new_name = f"({node_names[i]}:{d_iu:.4f},{node_names[j]}:{d_ju:.4f})"
        print(f"Step {step}: Join {node_names[i]} + {node_names[j]} "
              f"(Q={Q[i,j]:.2f}, branches={d_iu:.4f}/{d_ju:.4f})")
        
        # Step 4: Update distance matrix
        new_D = np.zeros((m - 1, m - 1))
        new_names = []
        idx_map = []
        
        for k in range(m):
            if k != i and k != j:
                new_names.append(node_names[k])
                idx_map.append(k)
        new_names.append(new_name)
        
        # Copy existing distances
        for a, ka in enumerate(idx_map):
            for b, kb in enumerate(idx_map):
                new_D[a, b] = D[ka, kb]
        
        # Distances to new node
        for a, ka in enumerate(idx_map):
            d_ku = 0.5 * (D[ka, i] + D[ka, j] - D[i, j])
            new_D[a, -1] = new_D[-1, a] = max(d_ku, 0.0)
        
        D = new_D
        node_names = new_names
    
    # Connect the final two nodes
    final_dist = D[0, 1]
    return f"({node_names[0]}:{final_dist/2:.4f},{node_names[1]}:{final_dist/2:.4f});"


print("Neighbor-Joining Tree Construction:")
print("=" * 60)
nj_newick = neighbor_joining(dm, seq_names)
print(f"\nResult: {nj_newick}")

In [ ]:
# Compare UPGMA and NJ on the same data
print("Comparison on the textbook example:")
print("=" * 60)
print("\n--- UPGMA ---")
upgma_result = upgma(example_dm, example_names)
print(f"Tree: {upgma_result}")

print("\n--- Neighbor-Joining ---")
nj_result = neighbor_joining(example_dm, example_names)
print(f"Tree: {nj_result}")

print("\nNote: UPGMA produces an ultrametric tree (equal-rate assumption).")
print("NJ allows unequal branch lengths, typically producing a more realistic tree.")

---

## 5. Character-Based Methods

Unlike distance methods which reduce sequences to a single number, **character-based methods** use the original alignment columns as data.

### 5.1 Maximum Parsimony

**Principle**: The best tree is the one requiring the **fewest evolutionary changes** (mutations) to explain the observed data.

**Algorithm:**
1. For each possible tree topology:
   - For each column in the alignment:
     - Calculate the minimum number of character changes needed (using Fitch's algorithm)
   - Sum changes across all columns = parsimony score
2. Choose the tree with the lowest total parsimony score

**Fitch's Algorithm** for a single site:
- At each leaf, assign the observed character as a set
- At each internal node: if children share characters, take the **intersection**; otherwise take the **union** and add 1 to the score

**Problem**: Searching all possible topologies is NP-hard. For $n$ taxa, there are $(2n-5)!!$ unrooted bifurcating trees.

| n taxa | Number of trees |
|--------|----------------|
| 4 | 3 |
| 5 | 15 |
| 10 | 2,027,025 |
| 20 | ~2.2 x 10^20 |
| 50 | ~2.8 x 10^74 |

In [ ]:
def fitch_parsimony_site(tree_node, site_index, alignment, names):
    """
    Fitch's algorithm for one alignment site.
    Returns the minimum number of changes and the ancestral state sets.
    """
    seq_map = dict(zip(names, alignment))
    
    def _fitch(node):
        if node.is_leaf():
            char = seq_map[node.name][site_index]
            return {char}, 0
        
        child_sets = []
        total_changes = 0
        
        for child in node.children:
            child_set, child_changes = _fitch(child)
            child_sets.append(child_set)
            total_changes += child_changes
        
        intersection = child_sets[0]
        for cs in child_sets[1:]:
            intersection = intersection & cs
        
        if intersection:
            return intersection, total_changes
        else:
            union = child_sets[0]
            for cs in child_sets[1:]:
                union = union | cs
            return union, total_changes + 1
    
    state_set, changes = _fitch(tree_node)
    return changes, state_set


def parsimony_score(tree_node, alignment, names):
    """
    Total parsimony score for a tree given an alignment.
    """
    aln_length = len(alignment[0])
    total = 0
    per_site = []
    
    for site in range(aln_length):
        changes, _ = fitch_parsimony_site(tree_node, site, alignment, names)
        total += changes
        per_site.append(changes)
    
    return total, per_site


# Build a simple tree for parsimony scoring
# Tree 1: ((A,B),(C,D))
tree1 = PhyloNode()
ab = tree1.add_child(PhyloNode())
ab.add_child(PhyloNode('A'))
ab.add_child(PhyloNode('B'))
cd = tree1.add_child(PhyloNode())
cd.add_child(PhyloNode('C'))
cd.add_child(PhyloNode('D'))

# Tree 2: ((A,C),(B,D))
tree2 = PhyloNode()
ac = tree2.add_child(PhyloNode())
ac.add_child(PhyloNode('A'))
ac.add_child(PhyloNode('C'))
bd = tree2.add_child(PhyloNode())
bd.add_child(PhyloNode('B'))
bd.add_child(PhyloNode('D'))

# Test alignment
test_aln = [
    "AATCCG",  # A
    "AATCCG",  # B
    "GGTCCA",  # C
    "GGTCCA",  # D
]
test_names = ['A', 'B', 'C', 'D']

score1, per_site1 = parsimony_score(tree1, test_aln, test_names)
score2, per_site2 = parsimony_score(tree2, test_aln, test_names)

print("Alignment:")
for name, seq in zip(test_names, test_aln):
    print(f"  {name}: {seq}")
print()
print(f"Tree 1: ((A,B),(C,D))  Parsimony score = {score1}  Per-site: {per_site1}")
print(f"Tree 2: ((A,C),(B,D))  Parsimony score = {score2}  Per-site: {per_site2}")
print(f"\nMost parsimonious tree: {'Tree 1' if score1 <= score2 else 'Tree 2'}")

### 5.2 Maximum Likelihood (ML)

**Principle**: Find the tree (topology + branch lengths + model parameters) that **maximizes the probability** of observing the data.

$$L(T, \theta | D) = P(D | T, \theta)$$

Where:
- $T$ = tree topology and branch lengths
- $\theta$ = substitution model parameters
- $D$ = observed alignment

For each column $i$ in the alignment:
$$L_i = \sum_{\text{ancestral states}} P(\text{data at tips} | \text{tree, model, ancestral states})$$

The total likelihood is the product across all independent sites:
$$L = \prod_i L_i$$

(In practice, we work with **log-likelihood** to avoid numerical underflow.)

**Advantages**: Statistically rigorous, can test models, handles rate variation

**Disadvantage**: Computationally intensive (heuristic tree search required)

**Tools**: RAxML, IQ-TREE, PhyML, MEGA

### 5.3 Bayesian Inference

**Principle**: Use Bayes' theorem to compute the **posterior probability** of a tree:

$$P(T | D) = \frac{P(D | T) \times P(T)}{P(D)}$$

Uses Markov Chain Monte Carlo (MCMC) to sample from the posterior distribution of trees.

**Advantages**: Provides true probability statements about trees, naturally handles uncertainty

**Disadvantage**: Slow; convergence can be difficult to assess

**Tools**: MrBayes, BEAST, RevBayes

In [ ]:
# Conceptual demonstration: likelihood for a simple 2-taxon case

def jukes_cantor_probability(from_base, to_base, branch_length):
    """
    Probability of substitution under JC69 model.
    P(same) = 1/4 + 3/4 * exp(-4t/3)
    P(diff) = 1/4 - 1/4 * exp(-4t/3)
    """
    if branch_length == 0:
        return 1.0 if from_base == to_base else 0.0
    
    exp_term = np.exp(-4 * branch_length / 3)
    if from_base == to_base:
        return 0.25 + 0.75 * exp_term
    else:
        return 0.25 - 0.25 * exp_term


# Show how likelihood varies with branch length
branch_lengths = np.arange(0.01, 2.0, 0.01)

# Pair of identical bases
p_same = [jukes_cantor_probability('A', 'A', t) for t in branch_lengths]
# Pair of different bases
p_diff = [jukes_cantor_probability('A', 'G', t) for t in branch_lengths]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(branch_lengths, p_same, 'b-', linewidth=2, label='P(A -> A) same base')
ax.plot(branch_lengths, p_diff, 'r-', linewidth=2, label='P(A -> G) different base')
ax.axhline(y=0.25, color='gray', linestyle='--', alpha=0.5, label='Equilibrium (1/4)')
ax.set_xlabel('Branch Length (expected substitutions/site)', fontsize=12)
ax.set_ylabel('Transition Probability', fontsize=12)
ax.set_title('JC69 Substitution Probabilities', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("At long branch lengths, all bases converge to 0.25 (saturation).")
print("This is why we cannot reliably estimate very large evolutionary distances.")

---

## 6. Bootstrap Analysis

Bootstrap analysis assesses the **statistical support** for each clade in a phylogenetic tree.

### Algorithm

```
Original alignment (L columns):
  Seq1: A T C G A T C G A T
  Seq2: A T C G T T C G A T
  Seq3: A T A G A T C G A T
  Cols: 1 2 3 4 5 6 7 8 9 10

Bootstrap replicate (sample L columns with replacement):
  Cols: 2 2 5 8 1 4 4 9 3 7
  Seq1: T T A G A G G A C C
  Seq2: T T T G A G G A C C
  Seq3: T T A G A G G A A C

Process:
  1. Generate 100-1000 bootstrap replicates
  2. Build a tree for each replicate
  3. For each clade in the original tree, count how often
     it appears in bootstrap trees
  4. Bootstrap support = percentage of trees containing the clade
```

### Interpretation

| Bootstrap Value | Interpretation |
|-----------------|----------------|
| >= 95% | Strong support |
| 70-95% | Moderate support |
| 50-70% | Weak support |
| < 50% | No support (clade is unreliable) |

In [ ]:
import random

def bootstrap_replicate(alignment):
    """
    Generate one bootstrap replicate by sampling columns with replacement.
    """
    seq_length = len(alignment[0])
    columns = [random.randint(0, seq_length - 1) for _ in range(seq_length)]
    
    return [''.join(seq[c] for c in columns) for seq in alignment]


def get_clades_from_newick(newick_str):
    """
    Extract all clades (sets of leaf names) from a Newick string.
    """
    tree = parse_newick(newick_str)
    clades = set()
    
    def _get_clades(node):
        if node.is_leaf():
            return frozenset([node.name])
        
        leaf_set = frozenset()
        for child in node.children:
            child_leaves = _get_clades(child)
            clades.add(child_leaves)
            leaf_set = leaf_set | child_leaves
        clades.add(leaf_set)
        return leaf_set
    
    _get_clades(tree)
    return clades


def bootstrap_analysis(alignment, names, n_replicates=100, tree_method='nj',
                       distance_method='jukes-cantor'):
    """
    Perform bootstrap analysis.
    
    Returns:
        original_tree: Newick string
        clade_support: dict mapping frozenset(leaf names) -> support percentage
    """
    # Build original tree
    orig_dm = build_distance_matrix(alignment, names, method=distance_method)
    if tree_method == 'nj':
        orig_tree = neighbor_joining(orig_dm, names)
    else:
        orig_tree = upgma(orig_dm, names)
    
    orig_clades = get_clades_from_newick(orig_tree)
    
    # Count clade support
    clade_counts = {clade: 0 for clade in orig_clades}
    
    for rep in range(n_replicates):
        boot_aln = bootstrap_replicate(alignment)
        boot_dm = build_distance_matrix(boot_aln, names, method=distance_method)
        
        try:
            if tree_method == 'nj':
                boot_tree = neighbor_joining(boot_dm, names)
            else:
                boot_tree = upgma(boot_dm, names)
            
            boot_clades = get_clades_from_newick(boot_tree)
            
            for clade in clade_counts:
                if clade in boot_clades:
                    clade_counts[clade] += 1
        except (ValueError, IndexError):
            continue  # Skip failed replicates
    
    clade_support = {clade: 100 * count / n_replicates 
                     for clade, count in clade_counts.items()}
    
    return orig_tree, clade_support


# Run bootstrap on our sequences (suppress step-by-step output)
import io, sys

# Temporarily suppress print output from NJ/UPGMA during bootstrap
old_stdout = sys.stdout
sys.stdout = io.StringIO()

random.seed(42)  # For reproducibility
orig_tree, support = bootstrap_analysis(
    aligned_seqs, seq_names, n_replicates=100, tree_method='nj'
)

sys.stdout = old_stdout

print(f"Original tree: {orig_tree}")
print(f"\nBootstrap support (100 replicates):")
for clade, pct in sorted(support.items(), key=lambda x: -x[1]):
    if len(clade) > 1 and len(clade) < len(seq_names):  # Skip trivial clades
        clade_str = ', '.join(sorted(clade))
        strength = "Strong" if pct >= 95 else "Moderate" if pct >= 70 else "Weak"
        print(f"  ({clade_str}): {pct:.0f}% [{strength}]")

---

## 7. Building Trees with BioPython (Bio.Phylo)

BioPython's `Bio.Phylo` module provides tools for reading, writing, and manipulating phylogenetic trees in various formats.

In [ ]:
from Bio import Phylo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Phylo.TreeConstruction import DistanceMatrix as BioDistanceMatrix
from Bio import AlignIO
from Bio.Align import MultipleSeqAlignment
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
import io

# Create an alignment for BioPython
bp_records = [
    SeqRecord(Seq("ATCGATCGATCGATCGATCG"), id="Sp_A"),
    SeqRecord(Seq("ATCGTTCGATCGATCGATCG"), id="Sp_B"),
    SeqRecord(Seq("TTCGATCGATCGATCGATCG"), id="Sp_C"),
    SeqRecord(Seq("TTCGTTCGATCGATCGATCG"), id="Sp_D"),
    SeqRecord(Seq("AACCGGTTAACCGGTTAACC"), id="Sp_E"),
]

bp_alignment = MultipleSeqAlignment(bp_records)
print("Alignment:")
for rec in bp_alignment:
    print(f"  {rec.id:>6}  {str(rec.seq)}")

In [ ]:
# Calculate distance matrix
calculator = DistanceCalculator('identity')
dm_bp = calculator.get_distance(bp_alignment)

print("BioPython Distance Matrix:")
print(dm_bp)

In [ ]:
# Build trees using BioPython's constructors
constructor = DistanceTreeConstructor()

# UPGMA tree
upgma_tree_bp = constructor.upgma(dm_bp)
print("UPGMA Tree:")
Phylo.draw_ascii(upgma_tree_bp)

print("\n" + "=" * 60)

# Neighbor-Joining tree
nj_tree_bp = constructor.nj(dm_bp)
print("\nNeighbor-Joining Tree:")
Phylo.draw_ascii(nj_tree_bp)

In [ ]:
# Tree properties
tree = nj_tree_bp

print("Tree Properties:")
print(f"  Number of terminals: {tree.count_terminals()}")
print(f"  Total branch length: {tree.total_branch_length():.4f}")
print(f"  Is bifurcating: {tree.is_bifurcating()}")

print(f"\nTerminal nodes (leaves):")
for terminal in tree.get_terminals():
    print(f"  {terminal.name}: branch_length = {terminal.branch_length:.4f}")

print(f"\nInternal nodes:")
for internal in tree.get_nonterminals():
    bl = f"{internal.branch_length:.4f}" if internal.branch_length else "None (root)"
    children = [c.name or 'internal' for c in internal.clades]
    print(f"  {internal.name or 'unnamed'}: branch_length = {bl}, children = {children}")

In [ ]:
# Distance between specific taxa
for name_a, name_b in [('Sp_A', 'Sp_B'), ('Sp_A', 'Sp_E'), ('Sp_C', 'Sp_D')]:
    d = tree.distance(name_a, name_b)
    print(f"  Distance {name_a} -> {name_b}: {d:.4f}")

# Find most recent common ancestor
mrca = tree.common_ancestor('Sp_A', 'Sp_B')
print(f"\nMRCA of Sp_A and Sp_B has {len(mrca.get_terminals())} descendants")
print(f"  Descendants: {[t.name for t in mrca.get_terminals()]}")

In [ ]:
# Read/write trees in different formats
# Write to Newick
newick_output = io.StringIO()
Phylo.write(nj_tree_bp, newick_output, 'newick')
print(f"Newick format: {newick_output.getvalue().strip()}")

# Parse from Newick string
newick_input = "((Human:0.1,Chimp:0.12):0.2,(Mouse:0.25,Rat:0.23):0.15);"
imported_tree = Phylo.read(io.StringIO(newick_input), 'newick')
print(f"\nImported tree:")
Phylo.draw_ascii(imported_tree)

---

## 8. Tree Visualization

Good visualization is key to interpreting phylogenetic results.

In [ ]:
def plot_tree_matplotlib(tree, title="Phylogenetic Tree"):
    """
    Plot a BioPython Phylo tree using matplotlib.
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    try:
        Phylo.draw(tree, axes=ax, do_show=False)
        ax.set_title(title, fontsize=13)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"matplotlib tree drawing failed: {e}")
        print("Falling back to ASCII:")
        Phylo.draw_ascii(tree)


# Plot the NJ tree
plot_tree_matplotlib(nj_tree_bp, title="Neighbor-Joining Tree")

In [ ]:
# Custom tree visualization from scratch
def draw_tree_custom(newick_str, title="Phylogenetic Tree"):
    """
    Draw a phylogenetic tree using custom matplotlib layout.
    """
    tree = parse_newick(newick_str)
    leaves = tree.get_leaves()
    n_leaves = len(leaves)
    
    # Assign y-positions to leaves
    leaf_y = {}
    for i, leaf in enumerate(leaves):
        leaf_y[id(leaf)] = i
    
    # Calculate x-positions (based on distance from root)
    def get_depth(node):
        if node.parent is None:
            return 0
        return get_depth(node.parent) + node.branch_length
    
    # Get y-position for internal nodes (mean of children)
    def get_y(node):
        if node.is_leaf():
            return leaf_y[id(node)]
        child_ys = [get_y(child) for child in node.children]
        return np.mean(child_ys)
    
    fig, ax = plt.subplots(figsize=(10, max(n_leaves * 0.5, 4)))
    
    def draw_node(node, x=0):
        y = get_y(node)
        
        if node.is_leaf():
            ax.plot(x, y, 'ko', markersize=5)
            ax.text(x + 0.005, y, f" {node.name}", va='center', fontsize=10)
        else:
            # Draw horizontal and vertical lines
            child_ys = []
            for child in node.children:
                child_x = x + child.branch_length
                child_y = get_y(child)
                child_ys.append(child_y)
                
                # Horizontal branch
                ax.plot([x, child_x], [child_y, child_y], 'k-', linewidth=1.5)
                
                draw_node(child, child_x)
            
            # Vertical connector
            ax.plot([x, x], [min(child_ys), max(child_ys)], 'k-', linewidth=1.5)
    
    draw_node(tree)
    
    ax.set_yticks([])
    ax.set_xlabel('Evolutionary Distance', fontsize=11)
    ax.set_title(title, fontsize=13)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    plt.tight_layout()
    plt.show()


# Draw our example tree
draw_tree_custom(
    "((Human:0.1,Chimp:0.12):0.2,(Mouse:0.25,Rat:0.23):0.15);",
    title="Custom Phylogram: Mammals"
)

In [ ]:
# Compare UPGMA vs NJ visually
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].set_title('UPGMA Tree', fontsize=13)
try:
    Phylo.draw(upgma_tree_bp, axes=axes[0], do_show=False)
except:
    axes[0].text(0.5, 0.5, 'UPGMA tree', ha='center', va='center', fontsize=14)

axes[1].set_title('Neighbor-Joining Tree', fontsize=13)
try:
    Phylo.draw(nj_tree_bp, axes=axes[1], do_show=False)
except:
    axes[1].text(0.5, 0.5, 'NJ tree', ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.show()

print("UPGMA: All tips equidistant from root (assumes molecular clock)")
print("NJ: Tips at different distances from root (allows rate variation)")

---

## 9. Interpreting Phylogenetic Trees

### 9.1 Common Misinterpretations

```
INCORRECT interpretation:           CORRECT interpretation:

  +--- Fish                          +--- Fish
  |                                  |
  +--- Frog    "Fish are more        +--- Frog    These are ALL equidistant
  |            primitive than"        |            from the root. You CAN'T
  +--- Lizard  "because they are     +--- Lizard  read left-to-right as a
  |            further left"          |            progression. The tree shows
  +--- Mouse                         +--- Mouse   branching order, not a
  |                                  |            ladder of complexity.
  +--- Human                         +--- Human
```

**Key rules for reading trees:**
1. **Tip order is arbitrary** -- you can rotate any node freely
2. **Proximity on the page does not mean close relatedness** -- only the branching pattern matters
3. **Branch lengths** (in phylograms) encode evolutionary change, not time unless ultrametric
4. **Internal nodes** represent hypothetical ancestors, not extant organisms

### 9.2 The Molecular Clock Hypothesis

The molecular clock hypothesis proposes that mutations accumulate at a roughly constant rate over time. If true:
- Branch lengths become proportional to time
- We can estimate divergence dates from sequence differences

In practice:
- **Strict clock**: Rarely holds; rates vary across lineages
- **Relaxed clock**: Allows rate variation; used by modern methods (BEAST)
- **Local clock**: Different parts of the tree evolve at different rates

### 9.3 Horizontal Gene Transfer (HGT)

Standard phylogenetics assumes **vertical inheritance** (parent to offspring). But HGT -- the transfer of genes between unrelated organisms -- is common in:
- Bacteria (conjugation, transformation, transduction)
- Archaea
- Eukaryotes (endosymbiosis, viruses)

HGT creates **discordance** between gene trees and species trees. If your gene tree contradicts the known species tree, HGT is a possible explanation.

### 9.4 Rooted vs Unrooted Trees

Most tree-building methods produce **unrooted** trees. To root a tree:
- **Outgroup rooting**: Include a known outgroup; the root lies on the branch connecting the outgroup
- **Midpoint rooting**: Place the root at the midpoint of the longest path
- **Molecular clock rooting**: If a clock holds, the root maximizes likelihood under a clock model

In [ ]:
# Demonstrate that tip order is arbitrary -- same tree, different appearance
print("These two trees have IDENTICAL topology:")
print()

tree_str1 = "((Human:0.1,Chimp:0.12):0.2,(Mouse:0.25,Rat:0.23):0.15);"
tree_str2 = "((Rat:0.23,Mouse:0.25):0.15,(Chimp:0.12,Human:0.1):0.2);"

t1 = Phylo.read(io.StringIO(tree_str1), 'newick')
t2 = Phylo.read(io.StringIO(tree_str2), 'newick')

print("Tree 1:")
Phylo.draw_ascii(t1)
print("\nTree 2 (same topology, rotated):")
Phylo.draw_ascii(t2)

print("Both trees show: (Human, Chimp) are sister taxa, and (Mouse, Rat) are sister taxa.")
print("The visual order on the page is meaningless!")

In [ ]:
# Outgroup rooting demonstration
def root_with_outgroup(tree, outgroup_name):
    """
    Root a tree using a specified outgroup.
    """
    tree_copy = tree  # Work with the tree directly
    try:
        tree_copy.root_with_outgroup(outgroup_name)
        return tree_copy
    except Exception as e:
        print(f"Rooting failed: {e}")
        return tree_copy


# Example: root the NJ tree with Sp_E as outgroup
print("Original NJ tree:")
Phylo.draw_ascii(nj_tree_bp)

print("\nRooted with Sp_E as outgroup:")
rooted_tree = nj_tree_bp
try:
    rooted_tree.root_with_outgroup('Sp_E')
    Phylo.draw_ascii(rooted_tree)
except Exception as e:
    print(f"Could not root: {e}")
    Phylo.draw_ascii(nj_tree_bp)

---

## 10. Robinson-Foulds Distance: Comparing Trees

How do we quantify the difference between two trees? The **Robinson-Foulds (RF) distance** counts the number of splits (bipartitions) present in one tree but not the other.

Each internal branch in an unrooted tree defines a **split**: it divides the taxa into two non-overlapping groups. The RF distance is the number of splits unique to each tree.

In [ ]:
def get_splits(newick_str):
    """
    Extract all bipartitions (splits) from a tree.
    A split is represented as a frozenset of leaf names on one side.
    """
    tree = parse_newick(newick_str)
    all_leaves = frozenset(n.name for n in tree.get_leaves())
    splits = set()
    
    def _collect_splits(node):
        if node.is_leaf():
            return frozenset([node.name])
        
        below = frozenset()
        for child in node.children:
            child_leaves = _collect_splits(child)
            below = below | child_leaves
        
        # A split is non-trivial if neither side is a single leaf or all leaves
        other_side = all_leaves - below
        if len(below) > 1 and len(other_side) > 1:
            # Canonical form: use the smaller set
            split = frozenset([below, other_side])
            splits.add(split)
        
        return below
    
    _collect_splits(tree)
    return splits


def robinson_foulds_distance(newick1, newick2):
    """
    Compute the Robinson-Foulds distance between two trees.
    """
    splits1 = get_splits(newick1)
    splits2 = get_splits(newick2)
    
    unique_to_1 = splits1 - splits2
    unique_to_2 = splits2 - splits1
    
    return len(unique_to_1) + len(unique_to_2)


# Compare trees
tree_a = "((A:0.1,B:0.2):0.3,(C:0.15,(D:0.1,E:0.1):0.2):0.1);"
tree_b = "((A:0.1,C:0.2):0.3,(B:0.15,(D:0.1,E:0.1):0.2):0.1);"
tree_c = "((A:0.1,B:0.2):0.3,(C:0.15,(D:0.1,E:0.1):0.2):0.1);"  # same as tree_a

print(f"Tree A: {tree_a}")
print(f"Tree B: {tree_b}")
print(f"Tree C: {tree_c}")
print()
print(f"RF distance (A vs B): {robinson_foulds_distance(tree_a, tree_b)}")
print(f"RF distance (A vs C): {robinson_foulds_distance(tree_a, tree_c)}")
print(f"RF distance (B vs C): {robinson_foulds_distance(tree_b, tree_c)}")

---

## 11. Simulating Evolution

Simulations let us test tree-building methods against a known truth. We evolve sequences along a known tree and check if our methods can recover the correct topology.

In [ ]:
def simulate_evolution(ancestor, tree_node, mutation_rate=0.05):
    """
    Simulate sequence evolution along a phylogenetic tree.
    
    Parameters:
        ancestor: starting DNA sequence string
        tree_node: PhyloNode tree
        mutation_rate: probability of mutation per site per unit branch length
    
    Returns:
        dict mapping leaf names to sequences
    """
    bases = 'ACGT'
    results = {}
    
    def _evolve(node, sequence):
        # Apply mutations proportional to branch length
        mutated = list(sequence)
        n_expected_mutations = int(len(sequence) * node.branch_length * mutation_rate)
        
        for _ in range(max(n_expected_mutations, 0)):
            pos = random.randint(0, len(mutated) - 1)
            current = mutated[pos]
            new_base = random.choice([b for b in bases if b != current])
            mutated[pos] = new_base
        
        mutated_seq = ''.join(mutated)
        
        if node.is_leaf():
            results[node.name] = mutated_seq
        else:
            for child in node.children:
                _evolve(child, mutated_seq)
    
    # Start from root
    if tree_node.is_leaf():
        results[tree_node.name] = ancestor
    else:
        for child in tree_node.children:
            _evolve(child, ancestor)
    
    return results


# Create a known tree
random.seed(42)
true_tree = PhyloNode()

ab_clade = true_tree.add_child(PhyloNode(branch_length=0.5))
ab_clade.add_child(PhyloNode('Species_A', 1.0))
ab_clade.add_child(PhyloNode('Species_B', 1.5))

cde_clade = true_tree.add_child(PhyloNode(branch_length=0.8))
cde_clade.add_child(PhyloNode('Species_C', 2.0))
de_clade = cde_clade.add_child(PhyloNode(branch_length=0.3))
de_clade.add_child(PhyloNode('Species_D', 1.0))
de_clade.add_child(PhyloNode('Species_E', 1.2))

# Simulate
ancestor = ''.join(random.choice('ACGT') for _ in range(200))
evolved = simulate_evolution(ancestor, true_tree, mutation_rate=0.1)

print("True tree: ((Species_A, Species_B), (Species_C, (Species_D, Species_E)))")
print(f"\nAncestor:    {ancestor[:60]}...")
for name, seq in sorted(evolved.items()):
    diffs = sum(a != b for a, b in zip(ancestor, seq))
    print(f"{name:>12}: {seq[:60]}... ({diffs} changes from ancestor)")

In [ ]:
# Can our methods recover the true tree?
sim_names = sorted(evolved.keys())
sim_seqs = [evolved[name] for name in sim_names]

sim_dm = build_distance_matrix(sim_seqs, sim_names, method='jukes-cantor')

print("Distance matrix from simulated sequences:")
print_distance_matrix(sim_dm, sim_names)

# Build NJ tree (suppress step output)
old_stdout = sys.stdout
sys.stdout = io.StringIO()
recovered_tree = neighbor_joining(sim_dm, sim_names)
sys.stdout = old_stdout

print(f"\nTrue topology: ((A, B), (C, (D, E)))")
print(f"Recovered NJ:  {recovered_tree}")
print("\nDoes the NJ method recover the correct groupings?")
print("Look for: A+B as sister taxa, and D+E as sister taxa.")

---

## 12. Real-World Example: Primate Phylogeny

Let us build a phylogeny from real mitochondrial cytochrome b sequences.

In [ ]:
# Cytochrome b fragments from primates (simplified, aligned)
primate_seqs = {
    'Human':      'ATGACCAACATCCGAAAATCACACCCACTAATAAAAATTGTAAACAACGCATTCATTGACCTCCCAGCCCCATCAAACATTTCATCATGATGAAACTTCGGCTCC',
    'Chimpanzee': 'ATGACCAACATCCGAAAATCACACCCACTAATAAAAATTGTAAACAACGCATTCATTGACCTCCCAGCTCCATCAAACATCTCATCATGATGAAACTTCGGCTCC',
    'Gorilla':    'ATGACCAACATCCGAAAATCACACCCACTAATAAAAATTGTAAACAACGCATTCATTGACCTCCCAGCCCCATCAAACATTTCGTCATGATGAAACTTCGGTTCC',
    'Orangutan':  'ATGACCAACATCCGGAAATCACACCCATTAATAAAAATTGTAAACAACGCATTTATTGACCTCCCAGCCCCATCAAACATTTCATCATGATGAAATTTTGGCTCC',
    'Gibbon':     'ATGACCAATATTCGAAAATCACACCCTCTAATAAAAATTGTAAACAACGCATTTATTGACCTTCCAGCTCCATCAAATATCTCATCATGATGAAATTTTGGCTCC',
    'Macaque':    'ATGACCAACATTCGAAAATCACATCCCCTAATAAAAATTGTAAATAACGCATTTATTGACCTTCCAACTCCATCAAATATTTCATCATGATGAAATTTTGGTTCC',
    'Mouse':      'ATGACAAACATCCGAAAATCTCACCCATTAATAAAGATTATTAACAACTCATTCATTGATCTTCCAACTCCATCTAACATCTCCGCATGATGAAATTTCGGTTCG',
}

# Build alignment
primate_records = [SeqRecord(Seq(seq), id=name) for name, seq in primate_seqs.items()]
primate_aln = MultipleSeqAlignment(primate_records)

print(f"Primate cytochrome b alignment: {len(primate_aln)} sequences, {primate_aln.get_alignment_length()} sites")
print()
for rec in primate_aln:
    print(f"  {rec.id:>12}: {str(rec.seq)[:60]}...")

In [ ]:
# Build trees using BioPython
calculator = DistanceCalculator('identity')
primate_dm = calculator.get_distance(primate_aln)
print("Distance matrix:")
print(primate_dm)

constructor = DistanceTreeConstructor()

# NJ tree
primate_nj = constructor.nj(primate_dm)
print("\nNeighbor-Joining Tree:")
Phylo.draw_ascii(primate_nj)

In [ ]:
# Root with Mouse as outgroup
primate_nj.root_with_outgroup('Mouse')

print("Rooted tree (Mouse outgroup):")
Phylo.draw_ascii(primate_nj)

# Visualize with matplotlib
fig, ax = plt.subplots(figsize=(10, 6))
try:
    Phylo.draw(primate_nj, axes=ax, do_show=False)
    ax.set_title('Primate Phylogeny (NJ, cytochrome b)', fontsize=13)
    plt.tight_layout()
    plt.show()
except Exception:
    print("(matplotlib visualization requires graphics backend)")

In [ ]:
# Compare with known primate phylogeny
print("Expected primate relationships:")
print("  Human and Chimpanzee are sister taxa (diverged ~6 MYA)")
print("  Gorilla is the next closest relative (~8 MYA)")
print("  Great apes (Human+Chimp+Gorilla+Orangutan) form a clade (~12 MYA)")
print("  Gibbon diverged from great apes (~18 MYA)")
print("  Old World monkeys (Macaque) diverged ~25 MYA")
print("  Mouse is the outgroup (~90 MYA)")
print()
print("Does your tree match these expected relationships?")
print("With only 101 bp of cytochrome b, resolution may be limited.")

---

## 13. Exercises

### Exercise 1: Build a Tree from Scratch

Given the following distance matrix, build both UPGMA and NJ trees. Compare the topologies.

In [ ]:
# Exercise 1: Build trees from this distance matrix

exercise_dm = np.array([
    [0.0,  0.3,  0.5,  0.6,  0.7],
    [0.3,  0.0,  0.4,  0.65, 0.72],
    [0.5,  0.4,  0.0,  0.55, 0.62],
    [0.6,  0.65, 0.55, 0.0,  0.2],
    [0.7,  0.72, 0.62, 0.2,  0.0],
])
exercise_names = ['Cat', 'Dog', 'Bear', 'Eagle', 'Hawk']

# YOUR CODE HERE:
# 1. Apply UPGMA to get a tree
# 2. Apply NJ to get a tree
# 3. Do the topologies agree? Which groupings make biological sense?
# 4. Calculate Robinson-Foulds distance between the two trees

print("Exercise 1: Build UPGMA and NJ trees and compare them.")
print("Expected: Eagle and Hawk should be sister taxa (both birds).")
print("Expected: Cat, Dog, Bear are all mammals.")

### Exercise 2: Distance Model Comparison

Build trees using p-distance, JC69, and Kimura distances from the same alignment. Do the topologies differ?

In [ ]:
# Exercise 2: Compare distance models

model_test_seqs = [
    "ATCGATCGATCGATCGATCGATCGATCGATCG",  # Seq1
    "ATCGTTCGATCGATCGATCGATCGATCGATCG",  # Seq2 (1 transition)
    "TTCGATCGATCGATCGATCGAACGATCGATCG",  # Seq3 (2 transversions)
    "TTCGTTCGATCGATCGATCGAACGATCGATCG",  # Seq4
    "AACCGGTTAACCGGTTAACCGGTTAACCGGTT",  # Seq5 (very different)
]
model_names = ['S1', 'S2', 'S3', 'S4', 'S5']

# YOUR CODE HERE:
# 1. Build distance matrices using all three methods
# 2. Build NJ trees from each
# 3. Compare: do the topologies differ?
# 4. For which sequence pairs does the correction matter most?

print("Exercise 2: Compare p-distance, JC69, and Kimura trees.")
print("When do corrected distances make a difference?")

### Exercise 3: Bootstrap Confidence

Perform bootstrap analysis and identify which clades are well-supported.

In [ ]:
# Exercise 3: Bootstrap analysis

# Use the primate sequences from Section 12
# YOUR CODE HERE:
# 1. Run bootstrap_analysis() with 200 replicates
# 2. Which clades have strong support (>= 95%)?
# 3. Which clades have weak support (< 70%)?
# 4. Is the (Human, Chimpanzee) clade well-supported?
# 5. What happens with more replicates (500, 1000)?

print("Exercise 3: Bootstrap analysis of primate phylogeny.")
print("Determine which relationships are statistically confident.")

### Exercise 4: Parsimony Scoring

Compare three possible tree topologies using parsimony.

In [ ]:
# Exercise 4: Maximum parsimony

pars_alignment = [
    "AATCCGATCG",  # X
    "AATCCGATCG",  # Y
    "GGTCCGATCA",  # Z
    "GGTAAGATCA",  # W
]
pars_names = ['X', 'Y', 'Z', 'W']

# YOUR CODE HERE:
# 1. Build all 3 possible unrooted topologies for 4 taxa:
#    - ((X,Y),(Z,W))
#    - ((X,Z),(Y,W))
#    - ((X,W),(Y,Z))
# 2. Calculate parsimony score for each
# 3. Which topology is most parsimonious?
# 4. Examine per-site scores: which positions are informative?

print("Exercise 4: Find the most parsimonious tree for 4 taxa.")
print("For 4 taxa, there are exactly 3 possible unrooted bifurcating topologies.")

### Exercise 5: Simulation Study

Use the evolution simulator to test when NJ outperforms UPGMA.

In [ ]:
# Exercise 5: When does NJ beat UPGMA?

# YOUR CODE HERE:
# 1. Create a true tree where branch lengths are VERY unequal
#    (violating the molecular clock assumption)
#    Example: ((A:0.1, B:0.5):0.2, (C:0.3, D:0.3):0.2)
# 2. Simulate sequences along this tree
# 3. Build both UPGMA and NJ trees from the simulated data
# 4. Compare recovered topologies to the true tree using RF distance
# 5. Repeat 50 times and count how often each method recovers the correct topology

print("Exercise 5: Simulation study comparing UPGMA vs NJ.")
print("Hypothesis: NJ should outperform UPGMA when rates vary across lineages.")

### Exercise 6: Interpret a Published Tree

Analyze and interpret the following tree from a hypothetical study.

In [ ]:
# Exercise 6: Tree interpretation

published_tree = "((((Human:0.01,Chimp:0.01)99:0.02,Gorilla:0.03)85:0.05,Orangutan:0.08)72:0.12,Macaque:0.20);"

tree_parsed = Phylo.read(io.StringIO(published_tree), 'newick')
print("Published tree:")
Phylo.draw_ascii(tree_parsed)

# YOUR ANSWERS HERE:
# 1. Which pair of taxa are most closely related?
# 2. What does the bootstrap value of 99 at the (Human, Chimp) node mean?
# 3. Why is the bootstrap value for the Orangutan clade only 72?
# 4. Which branch has the most evolutionary change?
# 5. Is this tree consistent with the known primate phylogeny?
# 6. If you wanted to add a Gibbon, where on the tree would it likely attach?

print("\nAnswer the questions about this tree in the cell below.")

---

## Summary

### Method Comparison

| Method | Type | Assumes Clock? | Complexity | Accuracy | Best For |
|--------|------|---------------|------------|----------|----------|
| **UPGMA** | Distance | Yes | O(n^3) | Low-Moderate | Guide trees for MSA |
| **Neighbor-Joining** | Distance | No | O(n^3) | Moderate | Quick exploratory analysis |
| **Maximum Parsimony** | Character | No | NP-hard | Moderate | Morphological data, few taxa |
| **Maximum Likelihood** | Statistical | Optional | Heuristic | High | Rigorous hypothesis testing |
| **Bayesian (MCMC)** | Statistical | Optional | Very slow | Highest | Divergence time estimation |

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Distance matrix** | Pairwise evolutionary distances between sequences |
| **JC69 / Kimura** | Models correcting for multiple substitutions |
| **UPGMA** | Assumes molecular clock; produces ultrametric trees |
| **Neighbor-Joining** | No clock assumption; most popular distance method |
| **Parsimony** | Minimizes total evolutionary changes |
| **Maximum Likelihood** | Maximizes probability of data given tree + model |
| **Bootstrap** | Resampling to assess clade confidence |
| **Robinson-Foulds** | Distance metric between tree topologies |
| **Outgroup rooting** | Using a known distant taxon to orient the tree |
| **Molecular clock** | Assumption of constant evolutionary rate |

### Best Practices

1. **Start with a good MSA** -- tree quality depends entirely on alignment quality
2. **Use NJ for quick exploration**, ML or Bayesian for publication
3. **Always report bootstrap values** -- a tree without support values is hard to interpret
4. **Include an outgroup** to root your tree
5. **Test multiple models** of sequence evolution (use model selection tools like jModelTest or ModelFinder)
6. **Be cautious with long branches** -- they can attract each other (Long Branch Attraction)
7. **Gene trees are not species trees** -- HGT, incomplete lineage sorting, and gene duplication complicate matters

---

## Resources

- [IQ-TREE](http://www.iqtree.org/) -- Fast maximum likelihood phylogenetics
- [RAxML](https://cme.h-its.org/exelixis/web/software/raxml/) -- ML tree inference for large datasets
- [MrBayes](http://nbisweden.github.io/MrBayes/) -- Bayesian phylogenetic inference
- [BEAST](https://beast.community/) -- Bayesian molecular dating
- [FigTree](http://tree.bio.ed.ac.uk/software/figtree/) -- Tree visualization
- [iTOL](https://itol.embl.de/) -- Interactive Tree of Life (online viewer)
- [MEGA](https://www.megasoftware.net/) -- Integrated phylogenetic analysis
- [Bio.Phylo docs](https://biopython.org/wiki/Phylo) -- BioPython phylogenetics module
- [The Phylogenetic Handbook](https://www.cambridge.org/core/books/the-phylogenetic-handbook/) -- Comprehensive reference (Lemey, Salemi, Vandamme)
- [Inferring Phylogenies](https://www.sinauer.com/inferring-phylogenies.html) -- Definitive text by Joe Felsenstein

---

[← Previous: Multiple Sequence Alignment: From Theory to Practice](../../Tier_2_Core_Bioinformatics/05_Multiple_Sequence_Alignment/01_multiple_sequence_alignment.ipynb) | [Next: Protein Structure Analysis →](../../Tier_2_Core_Bioinformatics/07_Protein_Structure/01_protein_structure.ipynb)